# Visual Module — Evaluation

Compares the two visual backends the module exposes:

| Backend | What it is |
|---|---|
| **Work-placement model** | `dima806/ai_vs_real_image_detection` + the `run_01_stage2A` int8 weight delta, fine-tuned on `combined_dataset`. The model the report's Stage-2 results use. |
| **Community Forensics** | `OwensLab/commfor-model-384`, official weights from Park & Owens (arXiv:2411.04125), used out of the box with no fine-tuning. `commfor-model-224` is the cheaper variant of the same architecture. |

Two test sets:

1. **Sample images** (`data/sample_images/`) — 12 images, ground truth in the filename prefix. Fast smoke test.
2. **`combined_dataset` test split** — 3,462 held-out images, the split the report's figures use.

Everything reusable lives in `visual_classifier.py` and `evaluation.py`; this notebook only orchestrates and displays.

In [ ]:
import os
import sys

import numpy as np
import pandas as pd
from PIL import Image

# Notebooks run from src/visual_module/, so bootstrap the project root first —
# module code imports through the `src.` prefix.
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from src.visual_module import (
    COMMFOR_MODEL_224,
    COMMFOR_MODEL_384,
    CommunityForensicsClassifier,
    VisualClassifier,
    get_delta_base_model,
)
from src.visual_module import evaluation as ev

print("Project root:", PROJECT_ROOT)

## Configuration

Edit this cell and run the notebook top to bottom.

In [ ]:
RUN_NAME = "run_07_visual_backend_comparison"

# Canonical fused decision threshold (David, 2026-07-19). Reporting each stream
# at the same threshold keeps these numbers comparable to what fusion sees.
THRESHOLD = 0.55

SAMPLE_DIR = os.path.join(PROJECT_ROOT, "data", "sample_images")
TEST_SPLIT_DIR = os.path.join(PROJECT_ROOT, "data", "visual", "combined_dataset", "test")
OUTPUT_DIR = os.path.join(PROJECT_ROOT, "src", "visual_module", "outputs")

VISUAL_DELTA_PATH = os.path.join(
    PROJECT_ROOT, "src", "visual_module", "fine_tuned_model_delta",
    "run_01_stage2A_ft_combined_weight_delta.pt",
)

# Set to an int to score only the first N images of the test split while
# iterating; None runs the full 3,462.
TEST_LIMIT = None

# The 224 checkpoint is the same architecture at lower resolution. Off by
# default — enable to check whether CF's misses are resolution-related.
INCLUDE_COMMFOR_224 = False

BATCH_SIZE = 16

## Load the classifiers

The base model is derived from the delta rather than hardcoded — `load_weight_delta`
raises on a mismatch, since a delta applied to the wrong base yields a model that is
neither the base nor the fine-tuned one.

In [ ]:
base_model = get_delta_base_model(VISUAL_DELTA_PATH)
print("Delta records base model:", base_model)

work_placement = VisualClassifier(
    model_name_or_path=base_model,
    delta_path=VISUAL_DELTA_PATH,
)
print("Work-placement model ready on", work_placement.device)

In [ ]:
commfor = CommunityForensicsClassifier(repo_id=COMMFOR_MODEL_384)
print(f"Community Forensics ready on {commfor.device} (input {commfor.input_size}px)")

classifiers = {
    "Work placement (fine-tuned ViT)": work_placement,
    "Community Forensics 384": commfor,
}

if INCLUDE_COMMFOR_224:
    classifiers["Community Forensics 224"] = CommunityForensicsClassifier(
        repo_id=COMMFOR_MODEL_224
    )

list(classifiers)

## Test 1 — sample images

Ground truth comes from the filename prefix (`{real|ai|deepfake}...`); `ai` and
`deepfake` are both the positive (AI-generated) class.

In [ ]:
sample_paths = sorted(
    os.path.join(SAMPLE_DIR, f)
    for f in os.listdir(SAMPLE_DIR)
    if not f.startswith(".")
)
sample_images = [Image.open(p) for p in sample_paths]
sample_labels = np.array([
    0 if os.path.basename(p).startswith("real") else 1 for p in sample_paths
])

sample_probs = {
    name: np.asarray(ev.ai_probabilities(clf, sample_images))
    for name, clf in classifiers.items()
}

samples_table = pd.DataFrame(
    {"image": [os.path.basename(p) for p in sample_paths],
     "truth": np.where(sample_labels == 1, "AI-Generated", "Real"),
     **{name: probs.round(4) for name, probs in sample_probs.items()}}
).set_index("image")

samples_table

In [ ]:
sample_metrics = {
    name: ev.metrics_at(probs, sample_labels, THRESHOLD)
    for name, probs in sample_probs.items()
}
pd.DataFrame(sample_metrics).T[["n", "auc", "accuracy", "precision", "recall", "f1"]]

## Test 2 — `combined_dataset` test split

Streams the arrow shards directly rather than `load_from_disk()`, which stalls
materialising the corpus from iCloud. Expect several minutes per classifier.

In [ ]:
split_probs, split_labels = {}, None

for name, clf in classifiers.items():
    print(f"Scoring {name}...")
    probs, labels = ev.score_split(
        clf, TEST_SPLIT_DIR, batch_size=BATCH_SIZE, limit=TEST_LIMIT
    )
    split_probs[name] = probs
    split_labels = labels   # identical ordering across classifiers
    print(f"  done: {len(probs)} images")

In [ ]:
split_metrics = {
    name: ev.metrics_at(probs, split_labels, THRESHOLD)
    for name, probs in split_probs.items()
}
pd.DataFrame(split_metrics).T[["n", "auc", "accuracy", "precision", "recall", "f1"]]

### Score distribution

Community Forensics is strongly bimodal and high-precision / low-recall: it is
confident when it fires, but a substantial share of AI images score near zero.
This is the behaviour that matters for fusion — it complements a high-recall
stream rather than replacing one.

In [ ]:
pd.DataFrame({
    name: {
        "real median": float(np.median(probs[split_labels == 0])),
        "AI median": float(np.median(probs[split_labels == 1])),
        "AI scoring < 0.05 (missed)": float((probs[split_labels == 1] < 0.05).mean()),
        "real scoring > 0.95 (false alarm)": float((probs[split_labels == 0] > 0.95).mean()),
    }
    for name, probs in split_probs.items()
}).T.round(4)

## Figures

In [ ]:
%matplotlib inline

ev.plot_confusion_matrices(
    split_metrics,
    output_path=os.path.join(OUTPUT_DIR, f"{RUN_NAME}_confusion_matrices.png"),
);

In [ ]:
ev.plot_roc(
    {name: (probs, split_labels) for name, probs in split_probs.items()},
    output_path=os.path.join(OUTPUT_DIR, f"{RUN_NAME}_roc.png"),
);

## Save results

In [ ]:
ev.save_results(
    {
        "threshold": THRESHOLD,
        "delta_base_model": base_model,
        "delta_path": os.path.relpath(VISUAL_DELTA_PATH, PROJECT_ROOT),
        "samples12": {
            "metrics": sample_metrics,
            "probs": {
                name: dict(zip(samples_table.index, probs.round(4).tolist()))
                for name, probs in sample_probs.items()
            },
        },
        "combined_test": split_metrics,
    },
    os.path.join(OUTPUT_DIR, f"{RUN_NAME}_eval_results.json"),
)